# 🤖 AI Engineering Fundamentals — Lezione 4
## Notebook Gruppo B

**ITS Novitas 4.0 | Giovedì 28/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [1]:
GRUPPO = "B"
MEMBRI = ["Anna", "Lorenzo L.", "Stefano", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

Gruppo B — Anna, Lorenzo L., Stefano


In [2]:
# ⚠️ Prima esecuzione: ChromaDB scarica Sentence Transformers (~90MB)
!pip install anthropic chromadb pypdf sentence-transformers -q

from google.colab import userdata
import anthropic, os, chromadb

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
chroma_client = chromadb.Client()

DOCUMENTO_WIDATA = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
"""

print("✅ Setup completato!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

---
## 🎯 Tema del Gruppo B: Chunking & Embedding

Esplorate come la dimensione dei chunk e l'overlap
impattano la qualità del retrieval.
Trovate i parametri ottimali per il documento WiData.

---
### Esercizio 1 — Confronto chunk_size *(guidato)*

Stessa domanda, stesso documento, tre dimensioni di chunk diverse.
I chunk recuperati sono diversi? Quale dimensione trova le informazioni più utili?

In [3]:
# Esercizio 1 — confronto chunk_size

def chunka_testo(testo, chunk_size=400, overlap=50):
    chunks = []
    start = 0
    while start < len(testo):
        chunk = testo[start:start+chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def crea_collection(nome, chunks):
    """Crea una collection ChromaDB con i chunk forniti."""
    try:
        chroma_client.delete_collection(nome)
    except:
        pass
    coll = chroma_client.create_collection(nome)
    coll.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])
    return coll

def cerca_in_collection(domanda, collection, n=2):
    risultati = collection.query(query_texts=[domanda], n_results=n)
    return risultati["documents"][0]

# Crea tre collection con dimensioni diverse
dimensioni = [100, 400, 800]
collections = {}

for dim in dimensioni:
  chunks = chunka_testo(DOCUMENTO_WIDATA, chunk_size=dim, overlap=20)
  collections[dim] = crea_collection(f"widata_{dim}", chunks)
  print(f"chunk_size={dim}: {len(chunks)} chunk generati")


print()

# Stessa domanda sulle tre collection
domanda = "Qual è l'autonomia della batteria del sensore XS200?"
print(f"❓ {domanda}\n")

for dim in dimensioni:
    print(f"{'='*50}")
    print(f"chunk_size = {dim}")
    # TODO: cercate nella collection corrispondente
    chunks_trovati = cerca_in_collection(domanda, collections[dim])
    for i, chunk in enumerate(chunks_trovati):
        print(f"Chunk {i+1}: {chunk}")
    print()

# Quale chunk_size trova le informazioni più utili? Perché?
# il chunk 2 con chunk size = 800 perchè è l'unico che trova l'informazione sull'autonomia della batteria


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 52.3MiB/s]


chunk_size=100: 18 chunk generati
chunk_size=400: 4 chunk generati
chunk_size=800: 2 chunk generati

❓ Qual è l'autonomia della batteria del sensore XS200?

chunk_size = 100
Chunk 1: ensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura te
Chunk 2: 
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è proge

chunk_size = 400
Chunk 1: 
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2
Chunk 2: ee (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widat

---
### Esercizio 2 — L'effetto dell'overlap *(guidato)*

Spezzate lo stesso testo con overlap=0 e overlap=100.
Trovate un caso in cui la frase chiave è a cavallo tra due chunk
e verificate che l'overlap la recuperi correttamente.

In [4]:

# Esercizio 2 — effetto dell'overlap

SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""


def chat(messaggio, system=None):

  params = {
  "model": "claude-haiku-4-5-20251001",
  "max_tokens": 500,
  "messages": messaggio
  }
  if system:
    params["system"] = system

  risposta = client.messages.create(**params)
  testo = risposta.content[0].text

  return testo




def chat_rag_con_storia(domanda, collection):
  """Chatbot con RAG + conversazione multi-turno."""
  # TODO:
  # 1. Recupera chunk rilevanti con cerca()
  chunks_rilevanti = cerca_in_collection(domanda, collection)
  contesto = "\n---\n".join(chunks_rilevanti)

  # 2. Costruisci il messaggio con contesto + domanda
  prompt_con_contesto = f"""Documenti di riferimento:
                                  {contesto}
                                      ---
                            Domanda dell'utente: {domanda}"""


  risposta = chat([{"role": "user", "content": prompt_con_contesto}], system=SYSTEM_WIDATA)

  return risposta






# Testo di test dove l'informazione chiave è a cavallo tra due chunk
testo_test = """Il sensore XS200 è resistente alle intemperie grazie alla classificazione
IP67 che garantisce impermeabilità completa e protezione dalla polvere.
La connettività LoRaWAN permette trasmissioni fino a 15km in campo aperto.
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato).

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
"""

# Chunk size piccolo per forzare lo spezzamento nel punto critico
CHUNK_SIZE = 300

# SENZA overlap
chunks_no_overlap = chunka_testo(testo_test, chunk_size=CHUNK_SIZE, overlap=0)
print("SENZA overlap:")
for i, c in enumerate(chunks_no_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print()

# CON overlap
chunks_overlap = chunka_testo(testo_test, chunk_size=CHUNK_SIZE, overlap=100)
print("CON overlap=100:")
for i, c in enumerate(chunks_overlap):
    print(f"  Chunk {i+1}: '{c}'")

print()

# TODO: create due collection e cercate 'impermeabilità LoRaWAN'
# Quale versione recupera meglio le informazioni?
coll_no = crea_collection("test_no_overlap", chunks_no_overlap)
coll_si = crea_collection("test_overlap", chunks_overlap)

#domanda_test = "Il sensore è impermeabile e come trasmette i dati?"
domanda_test_senza_overlap =  "Quali sono le caratteristiche della batteria del sensore XS200?"
domanda_test_con_overlap =  "Com'è la connessione del GATEWAY GW500?"


print(f"❓ {domanda_test_senza_overlap}")
print("\nSENZA overlap — chunk recuperati:\n")
# TODO: cercate in coll_no
chunk_no = cerca_in_collection(domanda_test_senza_overlap, coll_no, n=2)
print("Primo Chunk:\n", chunk_no[0])
print("Secondo Chunk:\n", chunk_no[1])
print("\n")
print(f"❓ {domanda_test_con_overlap}")
print("\nCON overlap — chunk recuperati:\n")
# TODO: cercate in coll_si
chunk_si = cerca_in_collection(domanda_test_con_overlap, coll_si, n=2)
print("Primo Chunk:\n", chunk_si[0])
print("Secondo Chunk:\n", chunk_si[1])



print(f"\n❓ Utente: {domanda_test_senza_overlap}")
risposta2 = chat_rag_con_storia(domanda_test_senza_overlap, coll_no)
print(f"🤖 Risposta Claude senza overlap\n: {risposta2}\n")

print(f"\n❓ Utente: {domanda_test_con_overlap}")
risposta1 = chat_rag_con_storia(domanda_test_con_overlap, coll_si)
print(f"🤖 Risposta Claude con overlap\n: {risposta1}")



# L'overlap fa la differenza in questo caso? Quando è più importante?
# Se il contesto della domanda si trova a cavallo fra 2 chunk:
#     senza overlap taglia le parole/frasi e non vengono trovate tutte le informazioni riguardo quell'argomento
#     con overloap parte delle frasi si trovano nei chunk successivi e claude riesce a rispondere col contesto completo

# In questo caso ha erroneamente anche considerato "LoRaWAN" parte del contesto della domanda sul GATEWAY GW500,
#                      mentre in realtà l'ha preso dal chunk precedente che faceva riferimento al sensore XS200

# Per migliorare la situzione si potrebbe considerare l'idea di prendere solo 1 chunk (il più rilevante) al posto di 2, o diminuire l'overlap/chunk_size

SENZA overlap:
  Chunk 1: 'Il sensore XS200 è resistente alle intemperie grazie alla classificazione
IP67 che garantisce impermeabilità completa e protezione dalla polvere.
La connettività LoRaWAN permette trasmissioni fino a 15km in campo aperto.
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il s'
  Chunk 2: 'ensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-I'
  Chunk 3: 'on 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Conne'
  Chunk 4: 'ssione cloud via Ethernet, WiFi

---
### Esercizio 3 — Trovare i parametri ottimali *(libero)*

Create un mini-benchmark: 5 domande sul documento WiData
con risposte attese note. Testate almeno 4 combinazioni
di chunk_size e overlap. Quale combinazione risponde
correttamente al maggior numero di domande?

In [5]:
# Esercizio 3 — benchmark parametri chunking

# Dataset di test: domanda + risposta attesa (parola chiave)
dataset = [
    {"domanda": "Qual è l'autonomia del sensore XS200?", "atteso": "2 anni"},
    {"domanda": "Quanti sensori gestisce il gateway GW500?", "atteso": "1000"},
    {"domanda": "Quanto costa il piano Pro di Xplore?", "atteso": "49"},
    {"domanda": "Qual è il numero di telefono del supporto?", "atteso": "079"},
    {"domanda": "Qual è la classificazione IP del sensore?", "atteso": "IP67"},
]

# Combinazioni da testare
configurazioni = [
    {"chunk_size": 100, "overlap": 10},
    {"chunk_size": 200, "overlap": 30},
    {"chunk_size": 400, "overlap": 50},
    {"chunk_size": 800, "overlap": 100},
]


risultati = {}

for conf in configurazioni:
    cs = conf["chunk_size"]
    ov = conf["overlap"]


    # TODO: create la collection con questa configurazione
    # Per ogni domanda nel dataset:
    #   - cercate i chunk rilevanti
    #   - verificate se la parola attesa è nei chunk recuperati
    # Contate quante domande trovano la risposta corretta





    nome_coll = f"bench_{cs}_{ov}"
    collection_bench = chroma_client.get_or_create_collection(
        name=nome_coll,
        metadata={"hnsw:space": "cosine"}
    )


    chunks = chunka_testo(DOCUMENTO_WIDATA, chunk_size=cs, overlap=ov)


    collection_bench.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )

    corrette = 0


    for item in dataset:

        trovati = collection_bench.query(query_texts=[item["domanda"]], n_results=2)
        testo_recuperato = " ".join(trovati["documents"][0])

        trovato = item["atteso"] in testo_recuperato
        if trovato:
            corrette += 1


    risultati[f"size={cs}, overlap={ov}"] = corrette




print("\n" + "=" * 60)
print("TABELLA RIASSUNTIVA")
print("=" * 60)
print(f"{'Configurazione':<25} {'Corrette/5'}")
print("-" * 60)


for config, corrette in risultati.items():
    print(f"{config:<25} {corrette}/5")

print("-" * 60, "\n")

print(f"Configurazione ottimale per WiData: chunk_size=400, overlap=50")
print("Motivazione: recupera bene le info, senza portare troppo contesto inutile a Claude")



TABELLA RIASSUNTIVA
Configurazione            Corrette/5
------------------------------------------------------------
size=100, overlap=10      3/5
size=200, overlap=30      3/5
size=400, overlap=50      4/5
size=800, overlap=100     5/5
------------------------------------------------------------ 

Configurazione ottimale per WiData: chunk_size=400, overlap=50
Motivazione: recupera bene le info, senza portare troppo contesto inutile a Claude


---
### Esercizio 4 — Chunking semantico *(libero)*

Il chunking a lunghezza fissa è semplice ma spezza le frasi.
Implementate una versione che spezza sui paragrafi naturali
del documento. Confrontate con il chunking a lunghezza fissa.

In [7]:
# Esercizio 4 — chunking semantico sui paragrafi


def chunka_per_paragrafi(testo, max_chunk_size=600):
    """
    Spezza il testo sui paragrafi (doppio newline).
    Se un paragrafo è troppo lungo, lo spezza ulteriormente.
    """

    # TODO: dividete il testo per \n\n
    # Per ogni paragrafo: se è più lungo di max_chunk_size, spezzatelo
    # altrimenti usatelo direttamente
    # Filtrate i chunk vuoti



    paragrafi = testo.split("\n\n")
    chunks = []

    for par in paragrafi:
        par = par.strip()


        if not par:
            continue


        if len(par) > max_chunk_size:
            sotto_chunks = chunka_testo(par, chunk_size=max_chunk_size, overlap=0)
            chunks.extend(sotto_chunks)
        else:
            chunks.append(par)


    return [c for c in chunks if c.strip()]


chunks_semantici = chunka_per_paragrafi(DOCUMENTO_WIDATA)
chunks_fissi = chunka_testo(DOCUMENTO_WIDATA, chunk_size=400, overlap=50)

print(f"Chunking fisso: {len(chunks_fissi)} chunk")
print(f"Chunking semantico: {len(chunks_semantici)} chunk")
print()

# Mostrate i chunk semantici
print("Chunk semantici:")
for i, c in enumerate(chunks_semantici):
    print(f"  Chunk {i+1} ({len(c)} char): {c}\n")

print()

# TODO: create due collection e confrontate la qualità del retrieval
# con le stesse 5 domande del benchmark
# Il chunking semantico è migliore? In quali casi?


coll_fisso = chroma_client.get_or_create_collection(
    name="confronto_fisso",
    metadata={"hnsw:space": "cosine"}
)

coll_fisso.add(
    documents=chunks_fissi,
    ids=[f"chunk_fisso_{i}" for i in range(len(chunks_fissi))]
)

coll_semantico = chroma_client.get_or_create_collection(
    name="confronto_semantico",
    metadata={"hnsw:space": "cosine"}
)

coll_semantico.add(
    documents=chunks_semantici,
    ids=[f"chunk_sem_{i}" for i in range(len(chunks_semantici))]
)


print(f"{'Domanda':<45} {'Fisso':<8} {'Semantico'}")
print("-" * 65)

corrette_fisso = 0
corrette_sem = 0

for item in dataset:
    trovato_f = coll_fisso.query(
        query_texts=[item["domanda"]],
        n_results=2
    )

    trovato_s = coll_semantico.query(
        query_texts=[item["domanda"]],
        n_results=2
    )


    testo_fisso = " ".join(trovato_f["documents"][0])
    testo_sem = " ".join(trovato_s["documents"][0])

    ok_fisso = "✅" if item["atteso"] in testo_fisso else "❌"
    ok_sem = "✅" if item["atteso"] in testo_sem else "❌"

    if ok_fisso == "✅":
        corrette_fisso += 1

    if ok_sem == "✅":
        corrette_sem += 1

    print(f"{item['domanda']:<45} {ok_fisso:<8} {ok_sem}")

print("-" * 65)
print(f"{'TOTALE':<45} {corrette_fisso}/5      {corrette_sem}/5")
print()


print("Commento finale:")
print("Il chunking semantico segue meglio le sezioni naturali del manuale,")
print("quindi tende a tenere insieme le informazioni dello stesso prodotto.")
print("Il chunking fisso invece è più semplice, ma può spezzare le frasi")
print("o dividere a metà una specifica tecnica importante.")

Chunking fisso: 4 chunk
Chunking semantico: 5 chunk

Chunk semantici:
  Chunk 1 (33 char): WiData Srl — Manuale Prodotti IoT

  Chunk 2 (456 char): SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica
e qualità dell'aria (CO2, PM2.5). Classificazione IP67: impermeabile e resistente alla polvere.
Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni. Connettività: LoRaWAN, NB-IoT, WiFi.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

  Chunk 3 (333 char): GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare. Temperatura operativa: -40°C a +70°C.

  Chunk 4 (380 char): PIATTAFORMA XPL

---
## 📊 Preparate la presentazione (5 slide)

1. **Chunk troppo piccoli vs ottimali vs grandi** — con i vostri esempi concreti
2. **L'effetto dell'overlap** — mostrate il caso della frase spezzata
3. **Il vostro benchmark** — tabella con i risultati delle 4 configurazioni
4. **Chunking semantico vs fisso** — differenze e quando usare quale
5. **La vostra raccomandazione** — parametri ottimali per WiData con motivazione

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*